<a href="https://colab.research.google.com/github/23subbhashit/NYC-Taxi-ETL/blob/main/ETL_Job.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from pyspark.sql import SparkSession

In [14]:
spark = SparkSession.builder \
    .appName("ETL_Pipeline_Project") \
    .getOrCreate()

spark

# Extract - from Parquet File

In [15]:
import os

os.listdir('/content/drive/MyDrive/Data')

['yellow_tripdata_2025-01.parquet']

In [16]:
df = spark.read.parquet("/content/drive/MyDrive/Data/yellow_tripdata_2025-01.parquet")

df.printSchema()
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [17]:
df.count() #

3475226

In [18]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

# Transform - Data Cleaning

In [19]:
# Remove invalid or missing values.
df_clean = df.dropna(subset=[
    "passenger_count",
    "trip_distance",
    "fare_amount"
])

In [20]:
# Filter Invalid Trips
df_filtered = df_clean.filter(
    (df_clean.trip_distance > 0) &
    (df_clean.fare_amount > 0)
)

In [21]:
from pyspark.sql.functions import unix_timestamp, col
# Create trip duration and average speed.
df_transformed = df_filtered.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") -
     unix_timestamp("tpep_pickup_datetime")) / 60
)

In [22]:
from pyspark.sql.functions import (
    unix_timestamp,
    col,
    try_divide
)
# Average Speed
df_transformed = df_transformed.withColumn(
    "avg_speed_kmh",
    try_divide(
        col("trip_distance") * 60,
        col("trip_duration_minutes")
    )
)

In [23]:
# Additional Feature (Hour of Day)
from pyspark.sql.functions import hour

df_transformed = df_transformed.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

In [24]:
# Data Aggregation
trip_summary = df_transformed.groupBy("pickup_hour").agg(
    {"trip_distance": "avg",
     "fare_amount": "avg"}
)

trip_summary.show()

+-----------+------------------+------------------+
|pickup_hour|  avg(fare_amount)|avg(trip_distance)|
+-----------+------------------+------------------+
|         12| 22.95872355181213|2.9509529543792223|
|         22|18.510513133826553| 3.517796538623919|
|          1| 17.15054727272728|3.2473456818181843|
|         13|18.041002548599746| 3.242176277134791|
|          6|  21.5842519259743| 5.832847030730318|
|         16| 18.79064142154571|3.2630350811525117|
|          3|17.229864745815444|3.2923796445043023|
|         20| 17.74468030240148| 3.318775488383216|
|          5|27.040971432264655|6.0348442347466325|
|         19| 17.04257006264435|3.0575472674147397|
|         15|18.726462329236163| 3.224370661779452|
|          9|17.113404095054754| 3.231641342555826|
|         17|  17.3580215025945|2.9611774682707415|
|          4| 23.15145885387284| 4.785629886354475|
|          8|16.992928279033297|2.9304846819566848|
|         23| 19.64055633365297| 3.929985888943892|
|          7

# Load - Save into a table

In [25]:
# Create Spark SQL Table
spark.sql("CREATE DATABASE IF NOT EXISTS taxi_db")

df_transformed.write \
    .mode("overwrite") \
    .saveAsTable("taxi_db.yellow_taxi_trips")

# Query the Table
spark.sql("""
SELECT pickup_hour,
       COUNT(*) as trips,
       AVG(fare_amount) as avg_fare
FROM taxi_db.yellow_taxi_trips
GROUP BY pickup_hour
ORDER BY pickup_hour
""").show()


+-----------+------+------------------+
|pickup_hour| trips|          avg_fare|
+-----------+------+------------------+
|          0| 65583|19.152312641995678|
|          1| 44000| 17.15054727272728|
|          2| 29671|16.203352768696682|
|          3| 19297|17.229864745815444|
|          4| 12407| 23.15145885387284|
|          5| 15472|27.040971432264655|
|          6| 35177|  21.5842519259743|
|          7| 74898|18.540995487196017|
|          8|106998|16.992928279033297|
|          9|123466|17.113404095054754|
|         10|134803|17.396891389657725|
|         11|145950| 17.22244385063397|
|         12|158612| 22.95872355181213|
|         13|167543|18.041002548599746|
|         14|180462| 18.63163159002999|
|         15|189882|18.726462329236163|
|         16|192354| 18.79064142154571|
|         17|208719|  17.3580215025945|
|         18|207653|16.356218450973607|
|         19|175914| 17.04257006264435|
+-----------+------+------------------+
only showing top 20 rows
